In [3]:
import time
import ipywidgets as widgets

from IPython.display import display, HTML, clear_output
from html import escape


# =========================================================
# 1. PEAS CHO MÁY HÚT BỤI SIMPLE AGENT
# =========================================================

PEAS = {
    "P - Performance Measure": [
        "Dọn sạch toàn bộ các ô trong môi trường.",
        "Số ô sạch càng nhiều càng tốt.",
        "Số bước thực hiện càng ít càng tốt.",
        "Không đi ra ngoài biên ma trận."
    ],
    "E - Environment": [
        "Môi trường là ma trận 3x3.",
        "Mỗi ô có thể là Dirty hoặc Clean.",
        "Agent bắt đầu tại ô (0,0)."
    ],
    "A - Actuators": [
        "SUCK: hút bụi tại ô hiện tại.",
        "UP: di chuyển lên.",
        "DOWN: di chuyển xuống.",
        "LEFT: di chuyển sang trái.",
        "RIGHT: di chuyển sang phải."
    ],
    "S - Sensors": [
        "Biết vị trí hiện tại của agent.",
        "Biết trạng thái ô hiện tại là Dirty hay Clean.",
        "Biết agent đang ở ô nào trong ma trận."
    ]
}


# =========================================================
# 2. CẤU HÌNH MÔI TRƯỜNG
# =========================================================

ROWS = 3
COLS = 3
MAX_STEPS = 30

START_LOCATION = (0, 0)


def create_start_grid():
    """
    Ban đầu tất cả ô đều bẩn.
    """
    return {
        (0, 0): "Dirty", (0, 1): "Dirty", (0, 2): "Dirty",
        (1, 0): "Dirty", (1, 1): "Dirty", (1, 2): "Dirty",
        (2, 0): "Dirty", (2, 1): "Dirty", (2, 2): "Dirty"
    }


def create_goal_grid():
    """
    Goal là tất cả ô đều sạch.
    """
    return {
        (0, 0): "Clean", (0, 1): "Clean", (0, 2): "Clean",
        (1, 0): "Clean", (1, 1): "Clean", (1, 2): "Clean",
        (2, 0): "Clean", (2, 1): "Clean", (2, 2): "Clean"
    }


START_GRID = create_start_grid()
GOAL_GRID = create_goal_grid()

current_grid = create_start_grid()
agent_location = START_LOCATION

step_count = 0
action_history = []

output = widgets.Output()


# =========================================================
# 3. HÀM XỬ LÝ MÔI TRƯỜNG
# =========================================================

def get_percept():
    """
    Percept = thứ agent cảm nhận được tại thời điểm hiện tại.

    Với máy hút bụi:
    percept = (vị trí hiện tại, trạng thái ô hiện tại)
    """
    global agent_location, current_grid

    location = agent_location
    status = current_grid[location]

    return location, status


def is_all_clean():
    """
    Kiểm tra toàn bộ môi trường đã sạch chưa.
    """
    return all(status == "Clean" for status in current_grid.values())


def clean_count(grid=None):
    """
    Đếm số ô sạch.
    """
    if grid is None:
        grid = current_grid

    return sum(1 for status in grid.values() if status == "Clean")


def get_valid_moves(location=None):
    """
    Trả về các hướng đi hợp lệ từ vị trí hiện tại.
    """
    if location is None:
        location = agent_location

    r, c = location
    moves = []

    if r > 0:
        moves.append("UP")

    if r < ROWS - 1:
        moves.append("DOWN")

    if c > 0:
        moves.append("LEFT")

    if c < COLS - 1:
        moves.append("RIGHT")

    return moves


def move_location(location, action):
    """
    Tính vị trí mới nếu di chuyển theo action.
    """
    r, c = location

    if action == "UP":
        return (r - 1, c)

    if action == "DOWN":
        return (r + 1, c)

    if action == "LEFT":
        return (r, c - 1)

    if action == "RIGHT":
        return (r, c + 1)

    return location


def execute_action(action):
    """
    Môi trường thực hiện action.
    """
    global agent_location, current_grid

    old_location = agent_location

    if action == "SUCK":
        current_grid[agent_location] = "Clean"

    elif action in ["UP", "DOWN", "LEFT", "RIGHT"]:
        if action in get_valid_moves(agent_location):
            agent_location = move_location(agent_location, action)

    return old_location, agent_location


def grid_to_text(grid, location=None):
    """
    In ma trận dạng text.
    Nếu có location, vị trí agent sẽ hiện dạng A-D hoặc A-C.
    """
    lines = []

    for r in range(ROWS):
        row = []

        for c in range(COLS):
            cell_location = (r, c)
            status = grid[cell_location]

            if location == cell_location:
                if status == "Dirty":
                    row.append("A-D")
                else:
                    row.append("A-C")
            else:
                if status == "Dirty":
                    row.append("D")
                else:
                    row.append("C")

        lines.append("  ".join(row))

    return "\n".join(lines)


# =========================================================
# 4. SIMPLE AGENT CHO MÁY HÚT BỤI
# =========================================================

class SimpleVacuumAgent:
    def choose_action(self, percept):
        """
        Simple Agent dùng rule:

        IF current cell is Dirty
        THEN SUCK

        ELSE:
            di chuyển theo đường quét cố định trong ma trận 3x3.

        Đường quét:
        (0,0) -> (0,1) -> (0,2)
                           ↓
        (1,0) <- (1,1) <- (1,2)
        ↓
        (2,0) -> (2,1) -> (2,2)
        """

        location, status = percept

        if is_all_clean():
            rule = "IF all cells are Clean THEN action = STOP"
            return "STOP", rule

        if status == "Dirty":
            rule = "IF current cell == Dirty THEN action = SUCK"
            return "SUCK", rule

        # Nếu ô hiện tại sạch thì đi theo rule quét cố định
        movement_rule = {
            (0, 0): "RIGHT",
            (0, 1): "RIGHT",
            (0, 2): "DOWN",
            (1, 2): "LEFT",
            (1, 1): "LEFT",
            (1, 0): "DOWN",
            (2, 0): "RIGHT",
            (2, 1): "RIGHT",
            (2, 2): "STOP"
        }

        action = movement_rule.get(location, "STOP")

        rule = """
IF current cell == Clean
THEN move according to fixed sweeping rule:
(0,0) -> RIGHT
(0,1) -> RIGHT
(0,2) -> DOWN
(1,2) -> LEFT
(1,1) -> LEFT
(1,0) -> DOWN
(2,0) -> RIGHT
(2,1) -> RIGHT
(2,2) -> STOP
"""

        return action, rule


agent = SimpleVacuumAgent()


# =========================================================
# 5. HTML GIAO DIỆN MA TRẬN
# =========================================================

def vacuum_html(grid, location):
    html = """
    <style>
        .vacuum-wrapper {
            font-family: Arial, sans-serif;
            margin-top: 10px;
        }

        .vacuum-title {
            font-size: 24px;
            font-weight: bold;
            margin-bottom: 12px;
        }

        .vacuum-grid {
            display: grid;
            grid-template-columns: repeat(3, 110px);
            grid-template-rows: repeat(3, 110px);
            gap: 6px;
            margin-bottom: 16px;
        }

        .cell {
            width: 110px;
            height: 110px;
            border: 2px solid #222;
            border-radius: 12px;
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            box-sizing: border-box;
            position: relative;
        }

        .dirty-cell {
            background: #ff9999;
        }

        .clean-cell {
            background: #b6f2b6;
        }

        .status-text {
            font-size: 18px;
            font-weight: bold;
            color: #111;
        }

        .position-text {
            margin-top: 8px;
            font-size: 13px;
            color: #333;
        }

        .agent-badge {
            position: absolute;
            top: 8px;
            right: 8px;
            background: yellow;
            color: black;
            border: 2px solid black;
            border-radius: 50%;
            width: 28px;
            height: 28px;
            display: flex;
            align-items: center;
            justify-content: center;
            font-size: 15px;
            font-weight: bold;
        }

        .info-box {
            background: #f7f7f7;
            color: #111;
            border-left: 5px solid #2196f3;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 820px;
        }

        .history-box {
            background: #fff8e1;
            color: #111;
            border-left: 5px solid #ff9800;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 820px;
            margin-top: 10px;
        }

        .peas-box {
            background: #e8f5e9;
            color: #111;
            border-left: 5px solid #4caf50;
            padding: 12px;
            border-radius: 8px;
            white-space: pre-wrap;
            font-family: Consolas, monospace;
            font-size: 14px;
            max-width: 820px;
            margin-top: 10px;
        }
    </style>

    <div class="vacuum-wrapper">
        <div class="vacuum-title">Vacuum Cleaner - Simple Agent</div>
        <div class="vacuum-grid">
    """

    for r in range(ROWS):
        for c in range(COLS):
            cell_location = (r, c)
            status = grid[cell_location]

            cell_class = "dirty-cell" if status == "Dirty" else "clean-cell"
            display_status = "Dirty" if status == "Dirty" else "Clean"

            agent_badge = ""
            if cell_location == location:
                agent_badge = "<div class='agent-badge'>A</div>"

            html += f"""
            <div class="cell {cell_class}">
                {agent_badge}
                <div class="status-text">{display_status}</div>
                <div class="position-text">({r},{c})</div>
            </div>
            """

    html += """
        </div>
    </div>
    """

    return html


def peas_text():
    text = "PEAS CHO MÁY HÚT BỤI SIMPLE AGENT\n\n"

    for key, values in PEAS.items():
        text += key + ":\n"
        for item in values:
            text += "- " + item + "\n"
        text += "\n"

    return text


# =========================================================
# 6. HÀM HIỂN THỊ
# =========================================================

def render(info_text="", show_peas=True):
    with output:
        clear_output(wait=True)

        display(HTML(vacuum_html(current_grid, agent_location)))

        basic_info = f"""
{info_text}

TRẠNG THÁI BAN ĐẦU:
{grid_to_text(START_GRID, START_LOCATION)}

TRẠNG THÁI HIỆN TẠI:
{grid_to_text(current_grid, agent_location)}

GOAL STATE:
{grid_to_text(GOAL_GRID, None)}

SỐ Ô SẠCH:
{clean_count(current_grid)} / 9
"""

        display(HTML(f"<div class='info-box'>{escape(basic_info)}</div>"))

        if len(action_history) == 0:
            history_text = "LỊCH SỬ ACTION:\nChưa có action nào."
        else:
            history_text = "LỊCH SỬ ACTION:\n" + "\n".join(action_history[-20:])

        display(HTML(f"<div class='history-box'>{escape(history_text)}</div>"))

        if show_peas:
            display(HTML(f"<div class='peas-box'>{escape(peas_text())}</div>"))


# =========================================================
# 7. CHƠI THỦ CÔNG
# =========================================================

def manual_action(action):
    global step_count

    if is_all_clean():
        render("Tất cả ô đã sạch. Agent đã hoàn thành nhiệm vụ.")
        return

    old_location = agent_location
    old_status = current_grid[old_location]
    old_clean = clean_count(current_grid)

    if action in ["UP", "DOWN", "LEFT", "RIGHT"]:
        if action not in get_valid_moves(old_location):
            render(f"Action {action} không hợp lệ tại vị trí {old_location}.")
            return

    before_location, after_location = execute_action(action)

    new_status = current_grid[after_location]
    new_clean = clean_count(current_grid)

    step_count += 1

    action_history.append(
        f"Manual Step {step_count} | State: {old_location} | Status: {old_status} | Action: {action} | New State: {after_location} | Clean: {old_clean}/9 -> {new_clean}/9"
    )

    info = f"""
MANUAL STEP {step_count}

State trước:
{old_location}

Trạng thái ô hiện tại:
{old_status}

Action người chơi chọn:
{action}

State sau:
{after_location}

Số ô sạch:
{old_clean}/9 -> {new_clean}/9
"""

    if is_all_clean():
        info += "\nOUTPUT CUỐI CÙNG: TẤT CẢ Ô ĐÃ SẠCH."

    render(info)


def on_suck_clicked(button):
    manual_action("SUCK")


def on_up_clicked(button):
    manual_action("UP")


def on_down_clicked(button):
    manual_action("DOWN")


def on_left_clicked(button):
    manual_action("LEFT")


def on_right_clicked(button):
    manual_action("RIGHT")


# =========================================================
# 8. SIMPLE AGENT CHẠY 1 BƯỚC
# =========================================================

def simple_agent_one_step(button=None):
    global step_count

    if is_all_clean():
        render("Tất cả ô đã sạch. Simple Agent hoàn thành nhiệm vụ.")
        return

    if step_count >= MAX_STEPS:
        render("Đã đạt MAX_STEPS. Simple Agent dừng để tránh chạy quá lâu.")
        return

    old_location = agent_location
    old_status = current_grid[old_location]
    old_clean = clean_count(current_grid)

    percept = get_percept()
    valid_moves = get_valid_moves(old_location)

    action, rule = agent.choose_action(percept)

    if action == "STOP":
        render("Simple Agent chọn STOP.")
        return

    before_location, after_location = execute_action(action)

    new_status = current_grid[after_location]
    new_clean = clean_count(current_grid)

    step_count += 1

    action_history.append(
        f"Simple Agent Step {step_count} | State: {old_location} | Percept: ({old_location}, {old_status}) | Action: {action} | New State: {after_location} | Clean: {old_clean}/9 -> {new_clean}/9"
    )

    info = f"""
SIMPLE AGENT STEP {step_count}

Percept agent nhận được:
{percept}

State trước:
{old_location}

Trạng thái ô hiện tại:
{old_status}

Các hướng di chuyển hợp lệ:
{valid_moves}

Rule được áp dụng:
{rule}

Action Simple Agent chọn:
{action}

State sau:
{after_location}

Số ô sạch:
{old_clean}/9 -> {new_clean}/9
"""

    if is_all_clean():
        info += "\nOUTPUT CUỐI CÙNG: TẤT CẢ Ô ĐÃ SẠCH."

    render(info)


# =========================================================
# 9. SIMPLE AGENT CHẠY TỰ ĐỘNG
# =========================================================

def simple_agent_auto_run(button=None):
    while not is_all_clean() and step_count < MAX_STEPS:
        simple_agent_one_step()
        time.sleep(0.4)

    if is_all_clean():
        render("OUTPUT CUỐI CÙNG: TẤT CẢ Ô ĐÃ SẠCH.")
    else:
        render("Simple Agent đã dừng vì đạt MAX_STEPS.")


# =========================================================
# 10. RESET
# =========================================================

def reset_environment(button=None):
    global current_grid, agent_location, step_count, action_history

    current_grid = create_start_grid()
    agent_location = START_LOCATION
    step_count = 0
    action_history = []

    render("Đã reset môi trường về trạng thái ban đầu.")


# =========================================================
# 11. BUTTON GIAO DIỆN
# =========================================================

suck_button = widgets.Button(
    description="SUCK",
    button_style="danger",
    layout=widgets.Layout(width="90px")
)

up_button = widgets.Button(
    description="UP",
    button_style="",
    layout=widgets.Layout(width="90px")
)

down_button = widgets.Button(
    description="DOWN",
    button_style="",
    layout=widgets.Layout(width="90px")
)

left_button = widgets.Button(
    description="LEFT",
    button_style="",
    layout=widgets.Layout(width="90px")
)

right_button = widgets.Button(
    description="RIGHT",
    button_style="",
    layout=widgets.Layout(width="90px")
)

agent_step_button = widgets.Button(
    description="Simple Agent 1 bước",
    button_style="success",
    layout=widgets.Layout(width="180px")
)

agent_auto_button = widgets.Button(
    description="Simple Agent tự chạy",
    button_style="info",
    layout=widgets.Layout(width="190px")
)

reset_button = widgets.Button(
    description="Reset",
    button_style="warning",
    layout=widgets.Layout(width="110px")
)

suck_button.on_click(on_suck_clicked)
up_button.on_click(on_up_clicked)
down_button.on_click(on_down_clicked)
left_button.on_click(on_left_clicked)
right_button.on_click(on_right_clicked)

agent_step_button.on_click(simple_agent_one_step)
agent_auto_button.on_click(simple_agent_auto_run)
reset_button.on_click(reset_environment)

manual_buttons = widgets.HBox([
    suck_button,
    up_button,
    down_button,
    left_button,
    right_button
])

agent_buttons = widgets.HBox([
    agent_step_button,
    agent_auto_button,
    reset_button
])


# =========================================================
# 12. HIỂN THỊ CHƯƠNG TRÌNH
# =========================================================

title = widgets.HTML(
    value="""
    <h2 style='font-family: Arial; margin-bottom: 5px;'>
        Vacuum Cleaner - Simple Agent
    </h2>
    <p style='font-family: Arial; font-size: 14px;'>
        Bấm các nút bên dưới để điều khiển thủ công hoặc cho Simple Agent chạy.
    </p>
    """
)

manual_label = widgets.HTML(
    value="<b style='font-family: Arial;'>Điều khiển thủ công:</b>"
)

agent_label = widgets.HTML(
    value="<b style='font-family: Arial;'>Điều khiển Simple Agent:</b>"
)

ui = widgets.VBox([
    title,
    manual_label,
    manual_buttons,
    agent_label,
    agent_buttons,
    output
])

display(ui)

render("""
MÔI TRƯỜNG MÁY HÚT BỤI BAN ĐẦU

Quy ước:
- Ô đỏ: Dirty = bẩn
- Ô xanh: Clean = sạch
- Huy hiệu A: vị trí agent máy hút bụi

Rule chính:
IF current cell == Dirty THEN action = SUCK
ELSE move according to fixed sweeping rule

Đường đi quét:
(0,0) -> (0,1) -> (0,2)
                  ↓
(1,0) <- (1,1) <- (1,2)
↓
(2,0) -> (2,1) -> (2,2)

Bấm 'Simple Agent 1 bước' để xem từng bước.
Bấm 'Simple Agent tự chạy' để chạy đến khi sạch hết.
""")